# Solvers and Jacobians

For a transition $f$, the equilibrium solves

$$
r(z)=f(z)-z=0.
$$

Local stability is governed by the state Jacobian

$$
J_z=\frac{\partial f}{\partial z}(z^\star).
$$

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})
from silva_networks import SolverConfig, fixed_point, full_jacobian, jvp, stability_report, vjp

torch.manual_seed(1)
W = 0.25 * torch.randn(4, 4)
b = torch.linspace(-0.2, 0.2, 4)

def f(z):
    return torch.tanh(W @ z + b)

z0 = torch.zeros(4)

In [ ]:
solver_runs = {}
for solver in ["picard", "anderson", "broyden"]:
    result = fixed_point(f, z0, SolverConfig(solver=solver, max_iter=20, alpha=0.6))
    solver_runs[solver] = result
    print(solver, result.iterations, result.residual)

In [ ]:
plt.figure(figsize=(5, 3))
for solver, result in solver_runs.items():
    plt.plot(result.residuals, marker="o", label=solver)
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("fixed-point residual")
plt.legend()
plt.tight_layout()

For small states, materialize $J_z$. For larger states, use products
$J_zv$ and $J_z^\top v$.

In [ ]:
result = fixed_point(f, z0, SolverConfig(solver="anderson", max_iter=20, alpha=0.6))
J = full_jacobian(f, result.z)
probe = torch.ones_like(result.z)
_, Jv = jvp(f, result.z, probe)
Jtv = vjp(f, result.z, probe)
print("J shape:", tuple(J.shape))
print("Jv:", Jv)
print("J^T v:", Jtv)

In [ ]:
report = stability_report(f, result.z, samples=4, iters=8)
report

## Citation and Sources

If this notebook or package is used, cite the software repository:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.0.0. MIT License.
https://github.com/jseluis/silva-networks
```

When the work is connected to the SILVA Networks paper, cite the paper as well:

```text
Jose Luis Lima de Jesus Silva. SILVA Networks as Structured Implicit Layers and
Vector Attractors via Dynamic Interaction Fields. 2026. arXiv:2607.28989.
https://arxiv.org/abs/2607.28989
```

Background references used in the tutorial suite include:

- Deep Equilibrium Models, Bai, Kolter, and Koltun, NeurIPS 2019:
  https://arxiv.org/abs/1909.01377
- Multiscale Deep Equilibrium Models, Bai, Koltun, and Kolter, NeurIPS 2020:
  https://arxiv.org/abs/2006.08656
- Stabilizing Equilibrium Models by Jacobian Regularization, Bai, Koltun, and
  Kolter, ICML 2021: https://arxiv.org/abs/2106.14342
- Graph Attention Networks, Velickovic et al., ICLR 2018:
  https://arxiv.org/abs/1710.10903
- Attention Is All You Need, Vaswani et al., 2017:
  https://arxiv.org/abs/1706.03762